# Imports

In [ ]:
import ollama
import sqlite3
from sentence_transformers import SentenceTransformer

import weaviate
from weaviate.classes.config import Property, DataType, Configure, VectorDistances
from weaviate.classes.query import MetadataQuery

# Configurações Iniciais

In [ ]:
WEAVITE_URL = "http://localhost:8080"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = 'llama3'

In [ ]:
print(f"Preparando Modelo de Embedding...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
encoder.encode('a').shape

In [ ]:
print(f"Conectando ao Banco Vetorial...")
weaviate_client = weaviate.connect_to_local()
print(weaviate_client.is_ready())

# Cria Collection

In [ ]:
def create_collection():

    if weaviate_client.collections.exists('ArtigoSuporte'):
        weaviate_client.collections.delete('ArtigoSuporte')

    weaviate_client.collections.create(
        name = 'ArtigoSuporte',
        properties = [
            Property(name='titulo', data_type=DataType.TEXT),
            Property(name='conteudo', data_type=DataType.TEXT),
            Property(name='sql_id', data_type=DataType.INT)
        ],
        vector_config=Configure.Vectors.self_provided(
            name = 'meu_vetor',
        vector_index_config = Configure.VectorIndex.hnsw(
            distance_metric = VectorDistances.COSINE
            )
        )
    )
    print(f"Collection criada com sucesso.")

# Ingestão de Dados: SQL → Weaviate

In [ ]:
def indexa_dados():

    # Conecta ao banco de dados
    conn = sqlite3.connect('suporte_tecnico.db') # banco de dados
    cursor = conn.cursor()

    # Executa consulta
    cursor.execute('SELECT id, titulo, conteudo FROM artigos_suporte') # tabela

    artigos = cursor.fetchall()

    collection = weaviate_client.collections.get("ArtigoSuporte")

    print(f"Iniciando indexação de {len(artigos)} artigos.")

    # Batch Insert

    with collection.batch.dynamic() as batch:

        for id, titulo, conteudo in artigos: # em artigo, id = sql_id, mas aqui eu nomeio do jeito que eu quiser (deixei id)

            # Gera o vetor de embedding
            texto_para_vetorizar = f"{titulo}: {conteudo}" # não vetoriza id
            vetor = encoder.encode(texto_para_vetorizar).tolist()

            batch.add_object(
                properties={
                    'titulo': titulo,
                    'conteudo': conteudo,
                    'sql_id': id
                },
                vector= {'meu_vetor': vetor}
            )
    conn.close()

    print(f"Indexação Concluída.")

# Busca Híbrida

In [ ]:
def busca_solucao(user_query:str, alpha:float):

    """
    alpha = 1.0 (busca vetorial).\n
    alpha = 0.0 (busca por palavra-chave)\n
    alpha = 0.5 (busca híbrida)
    """

    collection = weaviate_client.collections.get("ArtigoSuporte")

    vector_query = encoder.encode(user_query).tolist()

    response = collection.query.hybrid(
        query=user_query,
        alpha=alpha,
        vector=vector_query,
        limit=2,
        return_metadata=MetadataQuery(score=True)
    )

    return response.objects

In [ ]:
query_ = "Estou recebendo o Erro 503. O que fazer?"
context_ = busca_solucao(query_, 0.5)

In [ ]:
for obj_ in context_:
    print(obj_.properties['sql_id'])
    print(obj_.properties['titulo'])
    print(obj_.properties['conteudo'])
    print('\n---\n')

In [ ]:
# Isso é o que o LLM vai receber

context_str_ = "\n---\n".join([f"Título: {obj.properties['titulo']}\nConteúdo: {obj.properties['conteudo']}" for obj in context_])
print(context_str_)

# LLM

In [ ]:
def gera_resposta(user_query, context):

    # Formata o texto recuperado
    context_str = "\n---\n".join([f"Título: {obj.properties['titulo']}\nConteúdo: {obj.properties['conteudo']}" for obj in context])

    prompt = f"""
    Você é um assistente de suporte técnico Sênior. Use APENAS as informações abaixo para responder à pergunta do usuário.
    Se a informação não estiver no contexto, diga que não sabe.
    
    CONTEXTO RECUPERADO DO BANCO DE CONHECIMENTO:
    {context_str}
    
    PERGUNTA DO USUÁRIO:
    {user_query}
    
    RESPOSTA (Seja direto e técnico):
    """

    response = ollama.chat(model = LLM_MODEL, messages=[{'role': 'user', 'content': prompt}])
    
    return response['message']['content']

In [ ]:
create_collection()
indexa_dados()

while True:

    pergunta = input('Digite seu problema téncnico (ou sair)')
    if pergunta.lower() == 'sair':
        weaviate_client.close()
        break

    resultados = busca_solucao(pergunta, 0.5)

    if not resultados:
        print(f"Nenhum artigo relevante encontrado.")
        continue

    else:
        print(f"Encontrados {len(resultados)} artigos relevantes.")

    # Parte 1: Busca Híbrida

    for doc in resultados:
        if doc.metadata.score > 0.5:
            print(f" Título: {doc.properties['titulo']} | Score: {doc.metadata.score:.4f}")

    # Parte 2: Resposta do LLM

    resposta = gera_resposta(pergunta, resultados)
    print(f"\n RESPOSTA DO ASSISTENTE: {resposta}")

In [ ]:
# Qual a antecedência mínima para solicitação de férias?
# Estou recebendo o Erro 503. O que fazer?
# Minha consulta SQL está mais lenta do que o normal. O que pode causar isso?

In [285]:
weaviate_client.close()

# FIM